# 从相似度矩阵到视觉问答：CLIP 与 MLLM 的能力边界

本节实战以 **CLIP 为主线**，从一个图文相似度矩阵出发，依次完成：

1. CLIP 零样本图像分类；
2. CLIP 图文双向检索；
3. 冻结 CLIP，训练 Linear Probe；
4. 使用 Qwen3-VL 完成开放式视觉问答。

SigLIP 2 不作为另一套重复实验，而是作为 CLIP 的重要对照：我们会比较两者的训练目标、分数含义、预处理方式与适用场景。

> 这不是四段互不相关的 API 演示。贯穿全课的问题是：**一个相似度模型能完成什么？什么时候必须引入能够生成语言的多模态大模型？**


## 学习目标

完成本 Notebook 后，你应当能够：

- 解释 CLIP 的图像编码器、文本编码器、投影层与温度参数分别做什么；
- 手工重建 `归一化 → 点积 → 温度缩放 → softmax` 的零样本分类过程；
- 从相似度矩阵实现 Image-to-Text 与 Text-to-Image 的 Recall@K；
- 区分 zero-shot、linear probe 与 fine-tuning，理解它们分别测量什么；
- 说清 CLIP 的 batch-wise 对比学习与 SigLIP 的 pairwise sigmoid loss 的差异；
- 拆解 Qwen3-VL 的视觉预处理、视觉 token、chat template、prefill 与自回归解码；
- 用实验而不是印象判断 CLIP 与 MLLM 的能力边界。

建议课堂时长：3～4 小时。Qwen3-VL 与完整数据评测可根据显存和网速提前缓存。


## 官方依据与版本说明

本课代码优先沿用官方接口与示例：

- [OpenAI CLIP 官方仓库](https://github.com/openai/CLIP)：CIFAR-100 zero-shot 与 linear probe 示例；
- [OpenAI CLIP 官方交互 Notebook](https://github.com/openai/CLIP/blob/main/notebooks/Interacting_with_CLIP.ipynb)；
- [Transformers CLIP 文档](https://huggingface.co/docs/transformers/model_doc/clip)：用于统一处理器和模块级访问；
- [Transformers SigLIP 2 文档](https://huggingface.co/docs/transformers/model_doc/siglip2)；
- [Qwen3-VL 官方仓库](https://github.com/QwenLM/Qwen3-VL) 与 [Qwen3-VL-2B-Instruct 模型卡](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct)；
- [Flickr1K 图文检索数据](https://huggingface.co/datasets/nlphuji/flickr_1k_test_image_text_retrieval)：1,000 张测试图，每张图 5 条人工描述。

这里使用 Hugging Face Transformers 加载 OpenAI 权重，是为了让 CLIP、SigLIP 2 与 Qwen3-VL 共用一套接口；核心计算与官方 CLIP 示例一致。Qwen3-VL 官方要求 `transformers>=4.57.0`。


## 全课路线：同一张图，三种不同输出

| 模型 | 主要输出 | 必须预先给候选答案吗 | 典型用途 |
|---|---|---:|---|
| CLIP | 图文相似度 | 是 | 零样本分类、检索、过滤、表征 |
| SigLIP 2 | 每个图文对的匹配 logit | 是 | 检索、匹配、表征、定位等 |
| Qwen3-VL | 下一个 token 的概率分布 | 否 | 视觉问答、描述、OCR、推理、对话 |

CLIP/SigLIP 的共同核心是双塔编码：

```text
image ──> vision encoder ──> projection ──> v ─┐
                                                ├─> similarity matrix
text  ──> text encoder   ──> projection ──> t ─┘
```

Qwen3-VL 的核心是“把视觉表示接入语言模型上下文，再逐 token 生成”：

```text
image ─> vision encoder ─> visual merger/tokens ─┐
                                                  ├─> language model ─> token₁, token₂, ...
question ─> tokenizer ─> text tokens ────────────┘
```


## 0. 环境准备

推荐环境：

- CLIP 三个实验：8 GB 显存较舒适；CPU 也能运行课堂小子集，但提取特征会慢；
- SigLIP 2：课堂只跑小规模对照；
- Qwen3-VL-2B-Instruct：建议 8～12 GB 以上可用显存，具体取决于精度、图像尺寸和软件栈；CPU 可以加载但不适合课堂实时演示；
- 首次运行需要下载 CIFAR-100、Flickr1K 和模型权重。

如果在 Colab 中运行，安装依赖后建议重启一次 kernel。不要在课程开始后才第一次下载全部权重。


In [ ]:
# 若环境已经配置好，可以跳过此单元格。
# PyTorch 请按机器 CUDA 版本从 https://pytorch.org 安装；这里不强制重装 torch。
%pip install -qU "transformers>=4.57.0" "datasets>=3.0.0" accelerate \
    torchvision scikit-learn matplotlib seaborn pandas pillow tqdm


In [ ]:
from pathlib import Path
import gc
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR100

from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from transformers import (
    AutoModel,
    AutoModelForImageTextToText,
    AutoProcessor,
    CLIPModel,
    CLIPProcessor,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

WORK_DIR = Path.cwd() / "clip_mllm_lab"
DATA_DIR = WORK_DIR / "data"
CACHE_DIR = WORK_DIR / "feature_cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ===== 课堂开关 =====
QUICK_MODE = True                 # True: 小子集，适合课堂；False: 完整 CIFAR/Flickr1K
RUN_SIGLIP2 = True                # 下载并运行 SigLIP 2 小对照
RUN_QWEN3_VL = True               # 下载并运行 Qwen3-VL-2B-Instruct
RUN_NEXT_TOKEN_INSPECTION = True  # 多做一次 forward，观察“第一个回答 token”
RUN_VQA_SUITE = False             # True 会连续问多个问题，耗时明显增加

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
SIGLIP2_MODEL_ID = "google/siglip2-base-patch16-224"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
RETRIEVAL_DATASET_ID = "nlphuji/flickr_1k_test_image_text_retrieval"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("torch:", torch.__version__)
print("transformers device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### 通用观察工具

不要只看 `print(model)` 的几千行输出。下面两个工具分别回答：

1. 一个模块有多少参数，其中多少会被训练；
2. 模块层级与张量形状如何变化。


In [ ]:
def parameter_summary(module: nn.Module) -> dict:
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return {
        "total": total,
        "trainable": trainable,
        "total_M": round(total / 1e6, 2),
        "trainable_M": round(trainable / 1e6, 2),
    }


def shape_tree(obj):
    '''把 hook 的输入/输出压缩成容易阅读的 shape。'''
    if torch.is_tensor(obj):
        return tuple(obj.shape)
    if isinstance(obj, (list, tuple)):
        return [shape_tree(x) for x in obj]
    if isinstance(obj, dict):
        return {k: shape_tree(v) for k, v in obj.items()}
    if hasattr(obj, "to_tuple"):
        return shape_tree(obj.to_tuple())
    return type(obj).__name__


def module_children_table(module: nn.Module) -> pd.DataFrame:
    rows = []
    for name, child in module.named_children():
        info = parameter_summary(child)
        rows.append({"name": name, "type": type(child).__name__, **info})
    return pd.DataFrame(rows)


def move_batch(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


# 第一部分：拆开 CLIP，而不只调用 pipeline

CLIP 由两个 Transformer 编码器组成：

- **Vision Transformer**：图像被切成 patch，patch 经线性映射后与 class token、位置编码相加，再经过 self-attention；
- **Text Transformer**：文本经 BPE tokenizer 变成 token，加入位置编码并经过带 causal mask 的 self-attention，最终取 EOS 位置表示；
- **Projection**：两个塔的输出分别投影到相同维度；
- **Similarity**：L2 归一化后的向量点积就是余弦相似度，再乘可学习温度 `exp(logit_scale)`。

注意：CLIP 的 text tower 虽然使用 causal attention mask，但它不是在这里逐 token 生成回答；我们只取一个句向量做匹配。


In [ ]:
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(DEVICE).eval()
clip_device = DEVICE

print(parameter_summary(clip_model))
display(module_children_table(clip_model))


## 1.1 读配置：不要把模型结构写死在脑中

对 `ViT-B/32`：输入通常是 224×224，patch size 为 32，所以得到 $7\times7=49$ 个图像 patch；再加一个 class token，共 50 个视觉 token。文本最大长度为 77。两个塔的隐藏维度可以不同，但投影后维度相同。

请以实际配置输出为准，而不是背数字。


In [ ]:
vision_cfg = clip_model.config.vision_config
text_cfg = clip_model.config.text_config

clip_config_view = {
    "vision_image_size": vision_cfg.image_size,
    "vision_patch_size": vision_cfg.patch_size,
    "vision_hidden_size": vision_cfg.hidden_size,
    "vision_layers": vision_cfg.num_hidden_layers,
    "vision_heads": vision_cfg.num_attention_heads,
    "text_max_positions": text_cfg.max_position_embeddings,
    "text_hidden_size": text_cfg.hidden_size,
    "text_layers": text_cfg.num_hidden_layers,
    "text_heads": text_cfg.num_attention_heads,
    "shared_projection_dim": clip_model.config.projection_dim,
    "learned_logit_scale_exp": float(clip_model.logit_scale.exp().detach().cpu()),
}
display(pd.Series(clip_config_view, name="CLIP config"))

num_patches = (vision_cfg.image_size // vision_cfg.patch_size) ** 2
print(f"patch tokens = {num_patches}; 加 class token 后 = {num_patches + 1}")


## 1.2 预处理与 tokenization

模型看到的不是原始 PIL 图片或字符串：

- 图像会 resize / center crop，并按训练时的均值方差归一化；
- 文本会被 BPE tokenizer 切分，并加入起止 token；
- batch 内文本 padding 到同一长度，但不能超过 CLIP 的最大上下文长度。

下面先借用 CIFAR-100 的几张真实图片观察输入。


In [ ]:
cifar_train = CIFAR100(root=str(DATA_DIR), train=True, download=True)
cifar_test = CIFAR100(root=str(DATA_DIR), train=False, download=True)
class_names = [name.replace("_", " ") for name in cifar_test.classes]

demo_ids = [7, 42, 3637, 8888]
demo_images = [cifar_test[i][0] for i in demo_ids]
demo_targets = [cifar_test[i][1] for i in demo_ids]
demo_labels = [class_names[y] for y in demo_targets]
demo_texts = [f"a photo of a {label}" for label in demo_labels]

mini_inputs = clip_processor(
    images=demo_images,
    text=demo_texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

print("processor 输出：", {k: tuple(v.shape) for k, v in mini_inputs.items()})
print("第一条文本 token：", clip_processor.tokenizer.convert_ids_to_tokens(mini_inputs["input_ids"][0]))

fig, axes = plt.subplots(1, len(demo_images), figsize=(12, 3))
for ax, image, label in zip(axes, demo_images, demo_labels):
    ax.imshow(image)
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()


## 1.3 用 forward hook 追踪模块

hook 不改变模型，只记录中间模块的输入输出形状。重点观察：

- vision embeddings 输出的 token 数是否等于 patch 数 + 1；
- text embeddings 的 token 数是否等于当前 padding 后长度；
- 两个塔经过 projection 后是否进入相同维度。

Transformer 内部 attention 的概念计算为：

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M\right)V$$

多头 attention 将隐藏维拆成多个 head 并行计算，再拼接和线性映射。$M$ 在文本塔中包含 causal mask；视觉塔通常允许所有 patch 互相注意。


In [ ]:
captured = {}

def capture(name):
    def hook(module, inputs, output):
        captured[name] = {"input": shape_tree(inputs), "output": shape_tree(output)}
    return hook

handles = [
    clip_model.vision_model.embeddings.register_forward_hook(capture("vision_embeddings")),
    clip_model.vision_model.encoder.layers[0].self_attn.register_forward_hook(capture("vision_layer0_attention")),
    clip_model.text_model.embeddings.register_forward_hook(capture("text_embeddings")),
    clip_model.text_model.encoder.layers[0].self_attn.register_forward_hook(capture("text_layer0_attention")),
    clip_model.visual_projection.register_forward_hook(capture("visual_projection")),
    clip_model.text_projection.register_forward_hook(capture("text_projection")),
]

with torch.inference_mode():
    _ = clip_model(**move_batch(mini_inputs, clip_device))

for handle in handles:
    handle.remove()

display(pd.DataFrame(captured).T)


### 把 attention 公式翻译成张量操作

下面不是重写整个 CLIP，而是把一个多头 self-attention 的核心写出来。它帮助我们看清 `QKᵀ` 产生的是 token-token 权重矩阵；真正的官方前向仍由模型模块完成。


In [ ]:
def attention_core(q, k, v, additive_mask=None):
    '''
    q/k/v: [batch, heads, tokens, head_dim]
    additive_mask: 可选；允许位置为 0，被屏蔽位置为很大的负数
    '''
    head_dim = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / (head_dim ** 0.5)
    if additive_mask is not None:
        scores = scores + additive_mask
    weights = scores.softmax(dim=-1)
    context = weights @ v
    return context, weights

# 一个极小的形状演示：2 个 head，5 个 token，每个 head 4 维。
toy_q = torch.randn(1, 2, 5, 4)
toy_k = torch.randn(1, 2, 5, 4)
toy_v = torch.randn(1, 2, 5, 4)
toy_context, toy_weights = attention_core(toy_q, toy_k, toy_v)
print("context:", toy_context.shape, "attention matrix:", toy_weights.shape)
print("每一行权重和：", toy_weights[0, 0].sum(dim=-1))


## 1.4 手工重建 CLIP 的相似度矩阵

设图像向量为 $v_i$、文本向量为 $t_j$，归一化后：

$$S_{ij}=\exp(s)\cdot \frac{v_i}{\|v_i\|_2}^\top\frac{t_j}{\|t_j\|_2}$$

`logit_scale` 存的是 $s$，取指数后相当于逆温度。温度会改变 softmax 的尖锐程度，但不会改变单张图在固定文本集合上的排序。


In [ ]:
mini_inputs_device = move_batch(mini_inputs, clip_device)

with torch.inference_mode():
    official_outputs = clip_model(**mini_inputs_device)
    raw_image_features = clip_model.get_image_features(
        pixel_values=mini_inputs_device["pixel_values"]
    )
    raw_text_features = clip_model.get_text_features(
        input_ids=mini_inputs_device["input_ids"],
        attention_mask=mini_inputs_device["attention_mask"],
    )

image_features = F.normalize(raw_image_features, dim=-1)
text_features = F.normalize(raw_text_features, dim=-1)
temperature_scale = clip_model.logit_scale.exp()
manual_logits = temperature_scale * image_features @ text_features.T

print("image features:", image_features.shape)
print("text features:", text_features.shape)
print("similarity/logit matrix:", manual_logits.shape)
print(
    "与官方 forward 一致：",
    torch.allclose(manual_logits, official_outputs.logits_per_image, atol=1e-4, rtol=1e-4),
)

mini_probs = manual_logits.softmax(dim=-1).detach().cpu().numpy()
display(pd.DataFrame(mini_probs, index=demo_labels, columns=demo_labels).round(3))


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    manual_logits.detach().float().cpu().numpy(),
    annot=True,
    fmt=".1f",
    xticklabels=demo_labels,
    yticklabels=demo_labels,
    cmap="mako",
)
plt.xlabel("text candidates")
plt.ylabel("images")
plt.title("CLIP image-text logits")
plt.tight_layout()


# 实验一：CLIP 零样本分类

零样本分类并没有新增分类头。我们把每个类别写成文本 prompt，将所有类别文本向量当成“临时分类器权重”，再选择相似度最高者。

本实验不仅比较准确率，还考察 prompt template 是否影响结果。需要警惕：类别名和 prompt 都是模型输入的一部分，不是无关紧要的包装。


In [ ]:
@torch.inference_mode()
def build_clip_text_prototypes(class_names, templates):
    '''每个类别使用多个模板；模板内先归一化，再平均并再次归一化。'''
    prompts = [template.format(label) for label in class_names for template in templates]
    batch = clip_processor(
        text=prompts,
        padding=True,
        truncation=True,
        max_length=clip_model.config.text_config.max_position_embeddings,
        return_tensors="pt",
    )
    batch = move_batch(batch, clip_device)
    features = clip_model.get_text_features(**batch)
    features = F.normalize(features, dim=-1)
    features = features.view(len(class_names), len(templates), -1).mean(dim=1)
    return F.normalize(features, dim=-1)


@torch.inference_mode()
def encode_clip_images(dataset, indices, batch_size=128, normalize=True):
    '''直接从 PIL 图像提取 CLIP 特征；返回 CPU tensor 与标签。'''
    all_features, all_labels = [], []
    for start in tqdm(range(0, len(indices), batch_size), desc="encode images"):
        batch_ids = indices[start:start + batch_size]
        samples = [dataset[int(i)] for i in batch_ids]
        images = [sample[0] for sample in samples]
        labels = torch.tensor([sample[1] for sample in samples], dtype=torch.long)
        pixel_values = clip_processor(images=images, return_tensors="pt")["pixel_values"]
        raw = clip_model.get_image_features(pixel_values=pixel_values.to(clip_device))
        if normalize:
            raw = F.normalize(raw, dim=-1)
        all_features.append(raw.float().cpu())
        all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)


def topk_accuracy(logits, targets, k=1):
    topk = logits.topk(k, dim=-1).indices
    return topk.eq(targets[:, None]).any(dim=1).float().mean().item()


In [ ]:
rng = np.random.default_rng(SEED)
num_zero_shot_test = 1_000 if QUICK_MODE else len(cifar_test)
zero_shot_indices = rng.choice(len(cifar_test), size=num_zero_shot_test, replace=False).tolist()

single_template = ["a photo of a {}."]
prompt_ensemble = [
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a close-up photo of a {}.",
    "a photo of the small {}.",
    "a cropped photo of a {}.",
]

zs_image_features, zs_targets = encode_clip_images(
    cifar_test, zero_shot_indices, batch_size=128, normalize=True
)

prototype_sets = {
    "single template": build_clip_text_prototypes(class_names, single_template).cpu(),
    "prompt ensemble": build_clip_text_prototypes(class_names, prompt_ensemble).cpu(),
}

zero_shot_rows = []
zero_shot_logits = {}
scale = float(clip_model.logit_scale.exp().detach().cpu())
for name, prototypes in prototype_sets.items():
    logits = scale * zs_image_features @ prototypes.T
    zero_shot_logits[name] = logits
    zero_shot_rows.append({
        "setting": name,
        "samples": len(zs_targets),
        "top1": topk_accuracy(logits, zs_targets, 1),
        "top5": topk_accuracy(logits, zs_targets, 5),
    })

zero_shot_results = pd.DataFrame(zero_shot_rows)
display(zero_shot_results.style.format({"top1": "{:.2%}", "top5": "{:.2%}"}))


### 错误分析：不只报一个 accuracy

检查错误时至少区分三类问题：

- 图片本身低分辨率或主体不清；
- 预测类别与真实类别语义接近；
- prompt 对这个类别表达不自然。

下面随机展示错误样本。请先遮住真实标签，只看模型预测是否“完全荒谬”。


In [ ]:
best_name = "prompt ensemble"
predictions = zero_shot_logits[best_name].argmax(dim=-1)
wrong = torch.where(predictions != zs_targets)[0]

if len(wrong) > 0:
    chosen = wrong[torch.randperm(len(wrong))[:8]].tolist()
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, local_i in zip(axes.flat, chosen):
        dataset_i = zero_shot_indices[local_i]
        image, target = cifar_test[dataset_i]
        pred = int(predictions[local_i])
        ax.imshow(image)
        ax.set_title(f"true: {class_names[target]}\npred: {class_names[pred]}")
        ax.axis("off")
    plt.tight_layout()
else:
    print("当前子集没有错误样本。")


### 实验一讨论

请记录结果并回答：

1. prompt ensemble 是否一定优于单模板？如果不是，可能是什么原因？
2. softmax 概率是否等于“模型真的有 90% 把握”？候选类别集合改变时会发生什么？
3. 如果遗漏正确类别，CLIP 会拒答，还是仍从错误候选中选一个？
4. CIFAR-100 只有 32×32，CLIP 却按 224×224 处理，这可能造成什么损失？

可选扩展：自己设计 3～5 个模板，但在查看测试集结果前先固定模板；否则会把测试集变成调参集。


# 实验二：图文双向检索

分类是“一张图对固定类别文本”；检索则是“大量图片对大量自然语言描述”。两者都来自同一个相似度矩阵。

Flickr1K 每张图有 5 条人工描述，因此：

- Image-to-Text：一张图的 5 条描述都是正确答案；
- Text-to-Image：每条描述只对应它所在的那张图；
- 不能简单假设正确答案只在矩阵对角线上。

常用指标 Recall@K 表示：对每个 query，正确目标是否出现在前 K 个结果中，再对所有 query 求平均。


In [ ]:
try:
    flickr = load_dataset(RETRIEVAL_DATASET_ID, split="test")
except Exception as exc:
    raise RuntimeError(
        "Flickr1K 加载失败。请先升级 datasets，并确认能访问 Hugging Face。"
    ) from exc

num_retrieval_images = 100 if QUICK_MODE else len(flickr)
retrieval_ds = flickr.select(range(num_retrieval_images))

retrieval_images = [sample["image"].convert("RGB") for sample in retrieval_ds]
captions_per_image = [sample["caption"] for sample in retrieval_ds]
assert all(len(captions) == 5 for captions in captions_per_image)

# 展平为 5N 条文本，并记录每条文本属于哪张图。
retrieval_captions = [caption for captions in captions_per_image for caption in captions]
caption_to_image = torch.arange(num_retrieval_images).repeat_interleave(5)

print("images:", len(retrieval_images))
print("captions:", len(retrieval_captions))
print("columns:", retrieval_ds.column_names)
display(retrieval_ds[0]["image"])
print(*retrieval_ds[0]["caption"], sep="\n- ")


In [ ]:
@torch.inference_mode()
def encode_pil_images_with_clip(images, batch_size=64):
    chunks = []
    for start in tqdm(range(0, len(images), batch_size), desc="retrieval images"):
        batch = images[start:start + batch_size]
        pixels = clip_processor(images=batch, return_tensors="pt")["pixel_values"].to(clip_device)
        features = clip_model.get_image_features(pixel_values=pixels)
        chunks.append(F.normalize(features, dim=-1).float().cpu())
    return torch.cat(chunks)


@torch.inference_mode()
def encode_texts_with_clip(texts, batch_size=256):
    chunks = []
    max_len = clip_model.config.text_config.max_position_embeddings
    for start in tqdm(range(0, len(texts), batch_size), desc="retrieval texts"):
        batch = clip_processor(
            text=texts[start:start + batch_size],
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        batch = move_batch(batch, clip_device)
        features = clip_model.get_text_features(**batch)
        chunks.append(F.normalize(features, dim=-1).float().cpu())
    return torch.cat(chunks)


def bidirectional_recall(similarity, caption_to_image, ks=(1, 5, 10)):
    '''similarity: [num_images, num_captions]，分数越大越相似。'''
    num_images = similarity.shape[0]
    image_ranking = similarity.argsort(dim=1, descending=True)
    text_ranking = similarity.T.argsort(dim=1, descending=True)
    rows = []
    for k in ks:
        # 每张图有多条正确 caption：只要 top-k 中有任意一条属于该图即命中。
        i2t_hit = []
        for image_id in range(num_images):
            retrieved_caption_ids = image_ranking[image_id, :k]
            i2t_hit.append((caption_to_image[retrieved_caption_ids] == image_id).any())

        # 每条 caption 对应一张图。
        correct_images = caption_to_image
        t2i_hit = text_ranking[:, :k].eq(correct_images[:, None]).any(dim=1)
        rows.append({
            "K": k,
            "Image→Text Recall": torch.stack(i2t_hit).float().mean().item(),
            "Text→Image Recall": t2i_hit.float().mean().item(),
        })
    return pd.DataFrame(rows)


In [ ]:
retrieval_image_features = encode_pil_images_with_clip(retrieval_images)
retrieval_text_features = encode_texts_with_clip(retrieval_captions)
retrieval_similarity = retrieval_image_features @ retrieval_text_features.T

print("similarity matrix:", tuple(retrieval_similarity.shape))
retrieval_results = bidirectional_recall(
    retrieval_similarity, caption_to_image, ks=(1, 5, 10)
)
display(retrieval_results.style.format({
    "Image→Text Recall": "{:.2%}",
    "Text→Image Recall": "{:.2%}",
}))


### 可视化一个局部相似度矩阵

为了让矩阵可读，这里每张图只取第 1 条 caption。对角线较亮表示配对正确，但相似场景也可能产生高非对角分数。


In [ ]:
show_n = min(8, num_retrieval_images)
first_caption_ids = torch.arange(show_n) * 5
local_sim = retrieval_similarity[:show_n, first_caption_ids]
short_captions = [retrieval_captions[i][:38] + "…" for i in first_caption_ids]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.5]})

canvas = Image.new("RGB", (4 * 180, 2 * 180), "white")
for idx, image in enumerate(retrieval_images[:show_n]):
    thumb = image.copy()
    thumb.thumbnail((170, 170))
    x = (idx % 4) * 180
    y = (idx // 4) * 180
    canvas.paste(thumb, (x, y))
axes[0].imshow(canvas)
axes[0].set_title("query images: row-major order")
axes[0].axis("off")

sns.heatmap(
    local_sim.numpy(),
    ax=axes[1],
    cmap="mako",
    xticklabels=short_captions,
    yticklabels=[f"image {i}" for i in range(show_n)],
)
axes[1].tick_params(axis="x", rotation=70)
axes[1].set_title("CLIP cosine similarity")
plt.tight_layout()


In [ ]:
def show_text_to_image_retrieval(query_caption_id, top_k=5):
    scores = retrieval_similarity[:, query_caption_id]
    best = scores.topk(top_k).indices.tolist()
    correct = int(caption_to_image[query_caption_id])

    fig, axes = plt.subplots(1, top_k, figsize=(3 * top_k, 3.5))
    if top_k == 1:
        axes = [axes]
    for rank, (ax, image_id) in enumerate(zip(axes, best), start=1):
        ax.imshow(retrieval_images[image_id])
        mark = "✓" if image_id == correct else "✗"
        ax.set_title(f"#{rank} {mark}\nscore={scores[image_id]:.3f}")
        ax.axis("off")
    plt.suptitle(retrieval_captions[query_caption_id], y=1.06)
    plt.tight_layout()


show_text_to_image_retrieval(query_caption_id=min(12, len(retrieval_captions) - 1), top_k=5)


### 实验二讨论

1. 为什么 Image→Text 与 Text→Image 的 Recall 不一定相同？
2. 当数据量从 100 扩大到 1,000 时，Recall@1 为什么通常会下降？
3. 两条描述都能正确描述同一张图时，它们在文本空间中必须彼此接近吗？
4. 找一个失败案例：它是因为主体、动作、属性、数量还是空间关系被混淆？

进一步实验建议：

- 比较只用第一条 caption 与使用五条正例的评测差异；
- 比较 Recall@K、Mean Rank、Median Rank；
- 把 batch size、图像分辨率或模型骨干作为控制变量，而不是一次改变多个因素。


# 插曲：CLIP 与 SigLIP 2 到底哪里不同？

两者都能得到图文匹配分数，但训练目标不同。

## CLIP：batch 内多分类

对 batch 中 $B$ 个正确图文对，得到 $B\times B$ logits。每一行做“这张图对应哪条文本”的交叉熵，每一列再做“这段文本对应哪张图”的交叉熵。它强调 batch 内相对竞争。

## SigLIP：每个图文对独立二分类

对角图文对标为正，其余标为负，对每个 pair 使用 sigmoid/logistic loss。分数不需要在整行加和为 1，因此官方示例对 logits 使用 `sigmoid()`，而不是 `softmax()`。

SigLIP 2 延续 sigmoid loss，并加入更好的训练方法、数据与多语言能力等改进。这里使用 `google/siglip2-base-patch16-224` 做接口和分数解释对照，不重复跑所有大实验。


In [ ]:
def clip_symmetric_loss(logits):
    target = torch.arange(logits.shape[0], device=logits.device)
    image_to_text = F.cross_entropy(logits, target)
    text_to_image = F.cross_entropy(logits.T, target)
    return (image_to_text + text_to_image) / 2


def siglip_pairwise_loss(logits):
    # 对角为 +1（正对），非对角为 -1（负对）
    pair_labels = 2 * torch.eye(logits.shape[0], device=logits.device) - 1
    return F.softplus(-pair_labels * logits).mean()


toy_logits = torch.tensor([
    [4.0, 1.0, -1.0],
    [0.5, 3.0, 0.2],
    [-0.5, 0.1, 2.5],
])

print("CLIP symmetric CE:", float(clip_symmetric_loss(toy_logits)))
print("SigLIP pairwise logistic:", float(siglip_pairwise_loss(toy_logits)))
print("CLIP row softmax（每行和为 1）:\n", toy_logits.softmax(dim=-1))
print("SigLIP sigmoid（各 pair 独立）:\n", toy_logits.sigmoid())


## SigLIP 2 官方推理接口与预处理细节

官方文档特别强调：

- 文本使用类似 `This is a photo of {label}.` 的格式；
- 训练文本为小写，因此这里统一 `.lower()`；
- 使用 `padding="max_length", max_length=64`；
- `logits_per_image` 后接 sigmoid，得到每个图文 pair 的独立分数。

注意：sigmoid 输出不是天然校准的真实概率；阈值应在目标数据上验证。


In [ ]:
if RUN_SIGLIP2:
    siglip_processor = AutoProcessor.from_pretrained(SIGLIP2_MODEL_ID)
    siglip_model = AutoModel.from_pretrained(SIGLIP2_MODEL_ID).to(DEVICE).eval()

    print(parameter_summary(siglip_model))
    display(module_children_table(siglip_model))

    siglip_texts = [f"this is a photo of {label}.".lower() for label in demo_labels]
    siglip_inputs = siglip_processor(
        text=siglip_texts,
        images=demo_images,
        padding="max_length",
        max_length=64,
        return_tensors="pt",
    )
    siglip_inputs = move_batch(siglip_inputs, DEVICE)

    with torch.inference_mode():
        siglip_outputs = siglip_model(**siglip_inputs)
        siglip_scores = siglip_outputs.logits_per_image.sigmoid().float().cpu()

    display(pd.DataFrame(siglip_scores.numpy(), index=demo_labels, columns=demo_labels).round(3))
    print("raw logits shape:", tuple(siglip_outputs.logits_per_image.shape))
else:
    print("RUN_SIGLIP2=False：跳过权重下载。损失函数对照仍可运行。")


### CLIP / SigLIP 2 对照问题

1. softmax 改成 sigmoid 后，候选文本的数量会如何影响每个分数？
2. 如果同一张图同时符合两条描述，独立 sigmoid 与单行 softmax 哪个表达更自然？
3. 为什么不能直接比较 CLIP 的 `0.8` 与 SigLIP 的 `0.8`？
4. 模型、prompt、预处理与评测集都改变时，能否把性能差异归因于 loss？怎样设计更严谨的消融？

可选作业：复用实验二的编码与 Recall@K 框架，对同一 Flickr1K 子集运行 SigLIP 2；务必保持样本与指标完全一致。


# 实验三：冻结 CLIP，训练 Linear Probe

Zero-shot 测量“文本提示直接迁移”的能力；Linear Probe 测量“冻结视觉表征后，线性决策边界能提取多少任务信息”。

本实验严格冻结 CLIP，只缓存图像特征，再训练一个线性分类器。这样反向传播不会进入视觉编码器。

我们做两种实现：

1. scikit-learn Logistic Regression：对应 OpenAI CLIP 官方仓库示例；
2. PyTorch `nn.Linear`：显式观察 loss、optimizer 和参数更新。

官方示例中的 `C=0.316` 只是示范值；严谨实验应在验证集上搜索超参数，不能用测试集选择。


In [ ]:
# 明确冻结；此后 CLIP 不再产生梯度。
clip_model.requires_grad_(False)
clip_model.eval()
assert sum(p.numel() for p in clip_model.parameters() if p.requires_grad) == 0

rng = np.random.default_rng(SEED)
num_probe_train = 5_000 if QUICK_MODE else len(cifar_train)
num_probe_test = 1_000 if QUICK_MODE else len(cifar_test)
probe_train_indices = rng.choice(len(cifar_train), size=num_probe_train, replace=False).tolist()
# 与实验一使用同一批测试图片，避免把测试集差异误认为方法差异。
assert num_probe_test == len(zero_shot_indices)
probe_test_indices = list(zero_shot_indices)


def cached_clip_features(name, dataset, indices, batch_size=128):
    cache_path = CACHE_DIR / f"{name}_{CLIP_MODEL_ID.split('/')[-1]}_{len(indices)}_seed{SEED}.npz"
    if cache_path.exists():
        cached = np.load(cache_path)
        print("load cache:", cache_path)
        return cached["features"], cached["labels"]

    # Linear probe 通常使用投影后的原始特征；不做 L2 normalization。
    features, labels = encode_clip_images(
        dataset, indices, batch_size=batch_size, normalize=False
    )
    features_np = features.numpy()
    labels_np = labels.numpy()
    np.savez_compressed(cache_path, features=features_np, labels=labels_np)
    print("saved cache:", cache_path)
    return features_np, labels_np


train_features, train_labels = cached_clip_features(
    "cifar100_train", cifar_train, probe_train_indices
)
test_features, test_labels = cached_clip_features(
    "cifar100_test", cifar_test, probe_test_indices
)

print("train:", train_features.shape, train_labels.shape)
print("test:", test_features.shape, test_labels.shape)


In [ ]:
# 与 OpenAI CLIP 官方 linear-probe 示例保持相同的主要超参数。
sk_probe = LogisticRegression(
    random_state=SEED,
    C=0.316,
    max_iter=1_000,
    verbose=0,
)

start = time.time()
sk_probe.fit(train_features, train_labels)
sk_predictions = sk_probe.predict(test_features)
sk_accuracy = accuracy_score(test_labels, sk_predictions)

print(f"sklearn linear probe accuracy: {sk_accuracy:.2%}")
print(f"training time: {time.time() - start:.1f}s")
print("coefficient matrix:", sk_probe.coef_.shape)


## 3.1 手写 PyTorch Linear Probe 训练循环

这里训练的只有 $W,b$：

$$\hat y = Wx+b, \qquad \mathcal L=\operatorname{CrossEntropy}(\hat y,y)$$

CLIP 特征已经提前缓存并 `detach` 到 CPU，因此 optimizer 不可能更新 CLIP。这个实验不是 end-to-end fine-tuning。


In [ ]:
probe_device = DEVICE
x_train = torch.from_numpy(train_features).float()
y_train = torch.from_numpy(train_labels).long()
x_test = torch.from_numpy(test_features).float()
y_test = torch.from_numpy(test_labels).long()

probe_loader = DataLoader(
    TensorDataset(x_train, y_train),
    batch_size=256,
    shuffle=True,
)

linear_head = nn.Linear(x_train.shape[1], len(class_names)).to(probe_device)
optimizer = torch.optim.AdamW(linear_head.parameters(), lr=3e-3, weight_decay=1e-4)

history = []
for epoch in range(10):
    linear_head.train()
    running_loss = 0.0
    for features, labels in probe_loader:
        features = features.to(probe_device)
        labels = labels.to(probe_device)
        optimizer.zero_grad(set_to_none=True)
        logits = linear_head(features)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(labels)

    linear_head.eval()
    with torch.inference_mode():
        test_logits = linear_head(x_test.to(probe_device))
        test_acc = (test_logits.argmax(dim=-1).cpu() == y_test).float().mean().item()
    row = {
        "epoch": epoch + 1,
        "train_loss": running_loss / len(x_train),
        "test_accuracy_for_demo": test_acc,
    }
    history.append(row)
    print(row)

probe_history = pd.DataFrame(history)
probe_history.plot(x="epoch", y=["train_loss", "test_accuracy_for_demo"], subplots=True, figsize=(8, 5))
plt.tight_layout()


### 实验三结果解释

比较 zero-shot 与 linear probe 时，不要简单说“谁更强”：

- zero-shot 不使用 CIFAR-100 训练标签，测试的是语言提示迁移；
- linear probe 使用有标签训练数据，但不改变表征，测试的是线性可分性；
- full fine-tuning 会改变 encoder，训练成本和过拟合风险都更高。

课堂代码每轮显示 test accuracy 是为了观察趋势，不代表规范模型选择。正式实验应划分 train/validation/test，只用 validation 选 epoch、学习率与 `C`。

思考：如果 linear probe 大幅优于 zero-shot，问题可能出在视觉表征、文本类别表达，还是两者对齐？你会设计什么实验来区分？


In [ ]:
zero_shot_subset_top1 = float(
    zero_shot_results.loc[zero_shot_results["setting"] == "prompt ensemble", "top1"].iloc[0]
)
comparison = pd.DataFrame([
    {"method": "CLIP zero-shot", "uses CIFAR train labels": False, "updates CLIP": False, "accuracy": zero_shot_subset_top1},
    {"method": "sklearn linear probe", "uses CIFAR train labels": True, "updates CLIP": False, "accuracy": sk_accuracy},
    {"method": "PyTorch linear probe", "uses CIFAR train labels": True, "updates CLIP": False, "accuracy": history[-1]["test_accuracy_for_demo"]},
])
display(comparison.style.format({"accuracy": "{:.2%}"}))

print("注意：quick mode 下 zero-shot 与 probe 可能使用不同随机测试子集；正式比较应复用相同 test indices。")


# 实验四：Qwen3-VL 视觉问答

CLIP 能回答“这张图与哪条候选文本最像”，但不能自然地产生一段未预先枚举的回答。视觉问答需要把视觉信息与语言上下文融合，并进行自回归生成。

Qwen3-VL 官方仓库强调的架构更新包括：

- **Vision encoder / ViT**：把动态图像分辨率转换为视觉特征；
- **Visual merger**：压缩并映射视觉特征，使其进入语言模型隐藏空间；
- **DeepStack**：把多层视觉特征融合到语言模型不同层，增强细粒度视觉信息；
- **Interleaved-MRoPE**：联合表示时间、高度、宽度等位置信息；
- **Language model**：在视觉 token 与问题 token 条件下逐 token 预测回答。

本节选用官方 2B Instruct 权重，先用 CLIP 做固定候选匹配，再让 Qwen3-VL 回答同一张图，从输出形式上观察能力边界。


## 4.1 同一张官方示例图：先问 CLIP

CLIP 必须先提供候选答案。下面四个候选之外的答案，它不会主动生成。即使所有候选都错，它仍会给出最高分项。


In [ ]:
from transformers.image_utils import load_image

qwen_demo_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
qwen_demo_image = load_image(qwen_demo_url).convert("RGB")
display(qwen_demo_image)

clip_candidates = [
    "a person and a dog on a beach",
    "a crowded city street at night",
    "a plate of food on a table",
    "a document containing charts and text",
]

clip_question_inputs = clip_processor(
    images=[qwen_demo_image],
    text=clip_candidates,
    padding=True,
    return_tensors="pt",
)
clip_question_inputs = move_batch(clip_question_inputs, clip_device)

with torch.inference_mode():
    clip_candidate_logits = clip_model(**clip_question_inputs).logits_per_image
    clip_candidate_probs = clip_candidate_logits.softmax(dim=-1)[0].float().cpu()

clip_candidate_table = pd.DataFrame({
    "candidate": clip_candidates,
    "CLIP softmax within candidates": clip_candidate_probs.numpy(),
}).sort_values("CLIP softmax within candidates", ascending=False)
display(clip_candidate_table.style.format({"CLIP softmax within candidates": "{:.2%}"}))


## 4.2 加载 Qwen3-VL，并观察模块而不是打印整棵树

官方 Transformers 流程使用 `AutoModelForImageTextToText` 与 `AutoProcessor`。`device_map="auto"` 会根据可用设备放置权重。若显存不足，可在课前准备量化方案，但量化会引入额外依赖，不作为本 Notebook 的默认路径。

加载前把 CLIP 移回 CPU，给 Qwen3-VL 释放显存；前面已经缓存的实验结果不会丢失。


In [ ]:
if RUN_QWEN3_VL:
    clip_model.to("cpu")
    clip_device = torch.device("cpu")
    if RUN_SIGLIP2 and "siglip_model" in globals():
        siglip_model.to("cpu")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
    qwen_model = AutoModelForImageTextToText.from_pretrained(
        QWEN_MODEL_ID,
        dtype="auto",
        device_map="auto",
    ).eval()

    print(parameter_summary(qwen_model))
    display(module_children_table(qwen_model))
else:
    print("RUN_QWEN3_VL=False：跳过大模型下载。")


### 从配置定位视觉模块、合并比例与 DeepStack

不同 Transformers 版本中的 Python 类名可能变化，因此优先读 config，并用 `named_modules()` 搜索视觉相关模块，不把内部路径写死。

重点寻找：patch size、spatial merge、视觉层数、隐藏维度、DeepStack 层索引等字段。字段是否存在以当前权重配置为准。


In [ ]:
if RUN_QWEN3_VL:
    vision_config = getattr(qwen_model.config, "vision_config", None)
    if vision_config is not None:
        vision_dict = vision_config.to_dict()
        interesting_keys = [
            "depth", "num_hidden_layers", "hidden_size", "out_hidden_size",
            "num_attention_heads", "patch_size", "temporal_patch_size",
            "spatial_merge_size", "deepstack_visual_indexes",
        ]
        display(pd.Series({k: vision_dict.get(k) for k in interesting_keys if k in vision_dict}, name="vision config"))

    visual_modules = []
    for name, module in qwen_model.named_modules():
        depth = name.count(".")
        if depth <= 3 and any(key in name.lower() for key in ("visual", "vision", "merger")):
            visual_modules.append({
                "name": name,
                "type": type(module).__name__,
                "params_M": round(sum(p.numel() for p in module.parameters()) / 1e6, 2),
            })
    display(pd.DataFrame(visual_modules).drop_duplicates(subset=["name"]).head(30))


## 4.3 Chat template：文本是怎样“加进去”的？

`messages` 不是直接喂给神经网络的 Python 字典。Processor 会：

1. 按 chat template 插入 role、视觉占位符和 generation prompt；
2. tokenize 文本，生成 `input_ids`；
3. resize / normalize 图像并切成视觉 patch；
4. 返回图像网格信息，使视觉 token 能替换文本序列中的视觉占位位置。

`add_generation_prompt=True` 表示输入在 assistant 应开始回答的位置结束。


In [ ]:
question = "Describe this image, then explain the spatial relationship between the person and the dog."

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": qwen_demo_image},
            {"type": "text", "text": question},
        ],
    }
]

if RUN_QWEN3_VL:
    rendered_chat = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    print("===== chat template 渲染后的文本/占位符 =====")
    print(rendered_chat)

    qwen_inputs = qwen_processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    print("\n===== processor 输出张量 =====")
    for key, value in qwen_inputs.items():
        print(key, tuple(value.shape), value.dtype)


## 4.4 从 image grid 估算视觉 token 数

Qwen3-VL 支持动态分辨率，因此视觉 token 数不是固定常数。Processor 通常返回 `image_grid_thw = [T, H, W]`。若视觉 merger 的空间合并比例为 $m$，单图合并后的 token 数可近似理解为：

$$N_{visual}=T\times H\times W / m^2$$

最终应以 `input_ids` 中视觉占位 token 数和当前官方实现为准。更高图像分辨率通常意味着更多视觉 token、更高显存占用和更慢 prefill。


In [ ]:
if RUN_QWEN3_VL:
    grid = qwen_inputs.get("image_grid_thw")
    merge_size = getattr(getattr(qwen_model.config, "vision_config", None), "spatial_merge_size", None)
    image_token_id = getattr(qwen_model.config, "image_token_id", None)

    if grid is not None:
        print("image_grid_thw:", grid.tolist())
        if merge_size is not None:
            estimated = (grid.prod(dim=-1) // (merge_size ** 2)).tolist()
            print("按 spatial_merge_size 估算的 visual tokens:", estimated)

    if image_token_id is not None:
        count = (qwen_inputs["input_ids"] == image_token_id).sum(dim=-1).tolist()
        print("input_ids 中 image token 数:", count)
    else:
        print("当前 config 未直接暴露 image_token_id；请检查 tokenizer 的视觉特殊 token。")


## 4.5 自回归生成拆解：prefill 与 decode

第一次 forward（prefill）同时处理问题 token 和视觉 token，最后一个位置的 logits 给出“回答的第一个 token”的分布。之后的 decode 每次把新 token 接回上下文，并借助 KV cache 避免重复计算全部历史。

$$p(y\mid x, I)=\prod_{t=1}^{T}p(y_t\mid y_{<t},x,I)$$

下面先不调用 `generate()`，直接观察第一个 token 的候选。这一步与完整生成的首个 prefill 重复，因此可通过开关关闭。


In [ ]:
if RUN_QWEN3_VL and RUN_NEXT_TOKEN_INSPECTION:
    inspected_inputs = qwen_inputs.to(qwen_model.device)
    with torch.inference_mode():
        prefill_outputs = qwen_model(**inspected_inputs, use_cache=True)
    next_token_logits = prefill_outputs.logits[:, -1, :]
    next_token_probs = next_token_logits.softmax(dim=-1)
    top_probs, top_ids = next_token_probs[0].topk(10)

    next_token_table = pd.DataFrame({
        "token_id": top_ids.detach().cpu().tolist(),
        "token_text": [qwen_processor.tokenizer.decode([i]) for i in top_ids.detach().cpu().tolist()],
        "probability": top_probs.detach().float().cpu().tolist(),
    })
    display(next_token_table.style.format({"probability": "{:.3%}"}))

    # KV cache 是 decode 加速的关键；不同版本返回的具体类型可能不同。
    print("past_key_values type:", type(prefill_outputs.past_key_values).__name__)
    del prefill_outputs, inspected_inputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 4.6 官方 `generate()` 流程

下面保持 Qwen3-VL 官方 quickstart 的关键步骤：

1. `apply_chat_template(..., tokenize=True, return_dict=True)`；
2. 输入移到模型设备；
3. `model.generate()`；
4. 从输出中裁掉输入 token，只解码新生成部分。

`do_sample=False` 便于课堂复现；它不代表所有应用都应使用 greedy decoding。


In [ ]:
if RUN_QWEN3_VL:
    generation_inputs = qwen_inputs.to(qwen_model.device)
    with torch.inference_mode():
        generated_ids = qwen_model.generate(
            **generation_inputs,
            max_new_tokens=128,
            do_sample=False,
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(generation_inputs.input_ids, generated_ids)
    ]
    answers = qwen_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    print("Question:", question)
    print("Answer:", answers[0])


## 4.7 把官方流程封装成视觉问答函数

函数只是复用官方步骤，不隐藏关键中间量。课堂可用同一张图考察不同能力：属性、空间关系、计数、依据说明。不要只挑模型一定答对的问题。


In [ ]:
def ask_qwen3_vl(image, question, max_new_tokens=96):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question},
        ],
    }]
    inputs = qwen_processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(qwen_model.device)

    with torch.inference_mode():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    new_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    return qwen_processor.batch_decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]


if RUN_QWEN3_VL and RUN_VQA_SUITE:
    vqa_questions = [
        "What is the main scene? Answer in one sentence.",
        "Where is the dog relative to the person?",
        "How many visible animals are there? Give the number and brief evidence.",
        "What visual evidence suggests whether this is indoors or outdoors?",
    ]
    suite_answers = []
    for q in vqa_questions:
        a = ask_qwen3_vl(qwen_demo_image, q)
        suite_answers.append({"question": q, "answer": a})
    display(pd.DataFrame(suite_answers))
else:
    print("将 RUN_VQA_SUITE=True 后重新运行，可连续测试属性、空间、计数与证据问题。")


# 能力边界：不是“CLIP 弱、MLLM 强”这么简单

| 问题 | CLIP / SigLIP | Qwen3-VL |
|---|---|---|
| 在百万候选中快速检索 | 双塔向量可离线建索引，适合 | 每个 query 做生成通常太贵 |
| 固定标签零样本分类 | 直接、稳定、容易批量化 | 能回答，但输出约束与评测更复杂 |
| 开放式描述和问答 | 不能自由生成，只能匹配候选 | 原生适合 |
| 解释和多步语言推理 | 相似度本身不给解释 | 可以生成解释，但可能幻觉 |
| 分数解释 | 相对相似度；受候选集/温度影响 | token 概率；长答案由条件概率连乘 |
| 部署成本 | 编码后检索高效 | 参数、显存、延迟通常更高 |

常见工程组合不是二选一：先用 CLIP/SigLIP 召回少量候选，再把候选图交给 MLLM 做问答、重排或解释。


## 课堂综合挑战（保留探索空间）

选择一个方向，小组完成并用证据汇报：

### A. Prompt 与候选集

研究模板、类别同义词或候选类别数量如何改变 CLIP zero-shot。必须说明哪些设置在看测试结果前确定。

### B. 检索失败归因

从 Flickr1K 找出若干失败样本，自定一套错误类型；给出相似度、排名和图像证据，而不只展示“有趣案例”。

### C. Linear Probe 数据效率

改变有标签训练样本数量，画出 accuracy-data curve。保持 encoder、测试集和训练规则不变。

### D. CLIP 与 Qwen3-VL 边界

围绕同一组图片设计既包含固定候选、又包含开放回答的问题。先定义评价规则，再运行模型。

### E. SigLIP 2 对照

在相同 Flickr 子集上复用 Recall@K，讨论差异能否归因于损失函数；指出实验中仍未控制的变量。


## 实验记录模板

每次比较至少记录：

| 项目 | 内容 |
|---|---|
| 模型与 checkpoint | 例如 `openai/clip-vit-base-patch32` |
| 数据集与样本数 | quick/full、抽样 seed |
| 图像和文本预处理 | 分辨率、模板、截断长度 |
| 被训练参数 | none / linear head / encoder |
| 评价指标 | Top-1、Recall@K、人工规则等 |
| 单一自变量 | 本次究竟只改变了什么 |
| 失败案例 | 至少一个，并给出证据 |

如果一次同时改变模型规模、数据、prompt、预处理和 loss，就不能从结果中推出清晰因果结论。


## 结束前检查

- 我能否从两个归一化 embedding 手工构造 CLIP logits？
- 我是否知道 attention matrix 的两个 token 维分别代表什么？
- 我是否在有多个正确 caption 时正确实现 Recall@K？
- Linear Probe 的梯度有没有进入 CLIP？
- 为什么 SigLIP logits 应接 sigmoid 而非对候选做 softmax？
- Qwen3-VL 的图像经过哪些步骤才进入语言模型？
- `generate()` 返回的 token 为什么要裁掉输入部分？
- MLLM 生成了一段流畅解释，是否就证明解释真实？


In [ ]:
# 可选：课程结束后释放显存。
for name in ["qwen_model", "siglip_model", "clip_model"]:
    if name in globals():
        globals()[name].to("cpu")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("cleanup complete")


## 参考资料

1. Radford et al., *Learning Transferable Visual Models From Natural Language Supervision*, 2021.
2. Zhai et al., *Sigmoid Loss for Language Image Pre-Training*, 2023.
3. Tschannen et al., *SigLIP 2: Multilingual Vision-Language Encoders with Improved Semantic Understanding, Localization, and Dense Features*, 2025.
4. Qwen Team, *Qwen3-VL* 官方仓库与技术报告。
5. Hodosh et al., *Framing Image Description as a Ranking Task*, 2013（Flickr8K/1K 检索设置来源之一）。

官方代码与文档链接见 Notebook 开头。模型输出会随硬件、软件版本和抽样子集略有差异，结果解释应基于本次运行记录。
